# OOLONG-Synth task benchmark for Fleet

Load a reproducible task sample using Hugging Face Datasets and exercise the standard Fleet API. Default: twelve validation tasks covering task groups and context sizes, with at most two questions per context window. This is a diagnostic sample, not a leaderboard estimate.

Install notebook tooling with `uv sync --locked --group benchmarks` and select this repository's `.venv` kernel. The loader first projects small metadata columns, then retrieves unlabelled context only for the selected IDs. Gold answers remain host-side. The small attributed official scorer is in `scripts/oolong_scoring.py`; Fleet's metrics and judges remain in the product.

Historical data and measurements under `outputs/oolong/` are preserved. Database comparisons and special benchmark runtime profiles are retired.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
assert (ROOT / "pyproject.toml").is_file(), "Open this notebook from the Fleet repository"
sys.path.insert(0, str(ROOT / "scripts"))
from benchmark_tasks import freeze_tasks, load_tasks, run_tasks, score_tasks, evaluate_saved_tasks
from fleet_rlm.config import get_settings
from datasets import load_dataset
import pandas as pd

MLFLOW_MODE = "local"  # Or "databricks", matching the running Fleet service.
settings = get_settings(MLFLOW_MODE)
BASE_URL = f"http://{settings.host}:{settings.port}"
ALLOW_LIVE = False
ALLOW_JUDGES = False

In [ ]:
DATASET = "oolongbench/oolong-synth"
REVISION = "81d2f8d9349f6ff329d695f9be494051e9f05cb9"
SPLIT = "validation"
COUNT = 12
MAX_CONTEXT_TOKENS = 131072
SEED = 20261021
BUNDLE = ROOT / "outputs/benchmarks/oolong-validation-v1"
RUN_DIR = BUNDLE / "runs/run-001"

In [ ]:
from collections import Counter, defaultdict
from oolong_scoring import synth_process_response, UPSTREAM_COMMIT, UPSTREAM_BLOB


def select_oolong(rows, count=12, seed=20261021):
    groups = defaultdict(list)
    for row in rows:
        groups[(row["task_group"], int(row["context_len"]).bit_length())].append(row)
    for pool in groups.values():
        pool.sort(key=lambda row: hashlib.sha256(f"{seed}:{row['id']}".encode()).hexdigest())
    chosen, windows = [], Counter()
    while len(chosen) < count:
        added = False
        for key in sorted(groups):
            pool = groups[key]
            while pool and windows[pool[0]["context_window_id"]] >= 2:
                pool.pop(0)
            if pool and len(chosen) < count:
                row = pool.pop(0)
                chosen.append(row)
                windows[row["context_window_id"]] += 1
                added = True
        if not added:
            raise ValueError("Insufficient eligible tasks; refusing to shrink the sample")
    return chosen


def oolong_task(row):
    return {
        "case_id": f"oolong-{row['id']}",
        "task": "The complete unlabelled context is in files/context.txt. Read that file to answer the question below.\n\n"
        + row["question"],
        "files": {"context.txt": row["context_window_text"]},
        "source_group": f"context-window-{row['context_window_id']}",
        "expected_answer": row["answer"],
        "gold": {key: row[key] for key in ("id", "context_window_id", "dataset", "answer", "answer_type")},
    }


def official_score(case, answer):
    if (
        hashlib.sha256((ROOT / "scripts/oolong_scoring.py").read_bytes()).hexdigest()
        != bundle["metadata"]["verifier_sha256"]
    ):
        raise ValueError("Official scoring source changed since task freezing")
    return float(synth_process_response(case["gold"], answer, "fleet")["score"])

In [ ]:
if BUNDLE.exists():
    bundle = load_tasks(BUNDLE)
else:
    metadata_rows = load_dataset(
        DATASET,
        split=SPLIT,
        revision=REVISION,
        streaming=True,
        columns=["id", "task_group", "context_len", "context_window_id"],
        filters=[("context_len", "<=", MAX_CONTEXT_TOKENS)],
    )
    selected = select_oolong(metadata_rows, COUNT, SEED)
    selected_ids = [row["id"] for row in selected]
    rows = load_dataset(
        DATASET,
        split=SPLIT,
        revision=REVISION,
        streaming=True,
        columns=["id", "question", "context_window_text", "context_window_id", "dataset", "answer", "answer_type"],
        filters=[("id", "in", selected_ids)],
    )
    by_id = {row["id"]: row for row in rows}
    bundle = freeze_tasks(
        BUNDLE,
        [oolong_task(by_id[identity]) for identity in selected_ids],
        {
            "dataset": DATASET,
            "dataset_revision": REVISION,
            "split": SPLIT,
            "dataset_url": f"https://huggingface.co/datasets/{DATASET}/tree/{REVISION}",
            "count": COUNT,
            "max_context_tokens": MAX_CONTEXT_TOKENS,
            "seed": SEED,
            "verifier_sha256": hashlib.sha256((ROOT / "scripts/oolong_scoring.py").read_bytes()).hexdigest(),
            "verifier_commit": UPSTREAM_COMMIT,
            "verifier_blob": UPSTREAM_BLOB,
            "protocol": "Unlabelled context file, standard Fleet settings, official synth score",
        },
    )
display(pd.DataFrame([{"case_id": case["case_id"], "source_group": case["source_group"]} for case in bundle["cases"]]))
print("Frozen task digest:", bundle["digest"])

Run Fleet on the frozen tasks. Start the normal service yourself (`uv run fleet cli`, or `uv run fleet cli databricks`). Set `ALLOW_LIVE=True` only when ready for model and Daytona usage. The helper captures the effective configuration without changing models, mode, retries or token limits. Failed/uncertain receipts block replay; inspect their sessions and saved outcomes before choosing a new run.

In [ ]:
if ALLOW_LIVE:
    run_tasks(BUNDLE, RUN_DIR, base_url=BASE_URL, expected_destination=settings.mlflow_descriptor(), allow_live=True)
else:
    print("Live execution disabled; frozen task preparation and saved-output scoring are separate.")

Score existing receipts independently of execution. Official verification below is local, with model fallback disabled. Completed-case accuracy excludes operational failures; all-selected-case accuracy includes failures and unattempted cases in its denominator. Partial results are marked incomplete. OOLONG also has a fractional official score, reported separately from exact correctness.

In [ ]:
if (RUN_DIR / "run.json").exists():
    report = score_tasks(BUNDLE, RUN_DIR, official_score)
    display(pd.Series({key: value for key, value in report.items() if key != "cases"}))
    display(pd.json_normalize(report["cases"]))
else:
    print("No saved run yet.")

Evaluate the saved traces with Fleet's existing MLflow scorer suite and its DSPy judge, plus the official benchmark score as a separate assessment. This uses the configured evaluation experiment, records the source dataset and all selected task IDs, and labels traces `benchmark` so optimization curation excludes them. Set `ALLOW_JUDGES=True` deliberately: judges incur model usage. Trace IDs and the MLflow evaluation run remain available even if later scoring fails; re-score saved traces without executing tasks again.

In [ ]:
if ALLOW_JUDGES:
    evaluation = evaluate_saved_tasks(BUNDLE, RUN_DIR, settings, allow_judges=True)
    print("MLflow evaluation run:", evaluation.run_id)
    display(evaluation.result_df)
else:
    print("Fleet judge evaluation disabled.")